In [3]:
# 06 : Step 1: pool OOF predictions across folds
import sys, os, importlib
SCRIPTS = "/content/drive/MyDrive/0_potato_project_v1/scripts"

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path = [p for p in sys.path if p != SCRIPTS]
sys.path.insert(0, SCRIPTS)
importlib.invalidate_caches()

import numpy as np, pandas as pd
from pathlib import Path
import config as C

RUN_DIR = C.RESULTS / "runs" / "20260821_2337_baseline"   # <- edit if resuming later

oof = pd.concat(
    [pd.read_csv(RUN_DIR / f"fold_{k}" / "oof_preds.csv") for k in range(C.N_FOLDS)],
    ignore_index=True)

print("run       :", RUN_DIR.name)
print("pooled    :", oof.shape)
print("expected  : (2152, 8)")
print("unique    :", oof.path.nunique(), "paths |  duplicates:",
      len(oof) - oof.path.nunique())
print("per fold  :", oof.fold.value_counts().sort_index().tolist())
print("per class :", oof.y_true.value_counts().sort_index().tolist(),
      "->", [C.IDX_TO_CLASS[i] for i in range(3)])
print("probs ok  :", np.allclose(oof[["p0", "p1", "p2"]].sum(1), 1))
print("accuracy  :", f"{(oof.y_pred == oof.y_true).mean():.4f}")

run       : 20260821_2337_baseline
pooled    : (2152, 8)
expected  : (2152, 8)
unique    : 2152 paths |  duplicates: 0
per fold  : [429, 432, 431, 430, 430]
per class : [1000, 1000, 152] -> ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']
probs ok  : True
accuracy  : 0.9902


In [4]:
# 06 :  Step 2: confusion matrix + per-class metrics
from sklearn.metrics import confusion_matrix, classification_report, f1_score

y_true, y_pred = oof.y_true.values, oof.y_pred.values
names = [C.IDX_TO_CLASS[i].replace("Potato___", "") for i in range(C.NUM_CLASSES)]

cm = confusion_matrix(y_true, y_pred, labels=range(C.NUM_CLASSES))

print("CONFUSION MATRIX   (rows = true, cols = predicted)\n")
print(f"{'':>14}" + "".join(f"{n:>14}" for n in names))
for i, n in enumerate(names):
    print(f"{n:>14}" + "".join(
        f"{cm[i, j]:>14}" if i != j else f"{str(cm[i, j])+' ✓':>14}"
        for j in range(C.NUM_CLASSES)))

print("\n\nPER-CLASS METRICS\n")
print(classification_report(y_true, y_pred, target_names=names,
                            digits=4, zero_division=0))

print(f"pooled macro-F1 : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"per-fold mean   : 0.9804 ± 0.0063  (from summary.csv)")

print("\nERROR BREAKDOWN")
errs = oof[oof.y_pred != oof.y_true]
print(f"total errors    : {len(errs)} / {len(oof)}")
for (t, p), n in errs.groupby(["y_true", "y_pred"]).size().sort_values(ascending=False).items():
    print(f"  {names[t]:>13} -> {names[p]:<13} {n:>3}")

missed = len(errs[(errs.y_true != 2) & (errs.y_pred == 2)])
false_alarm = len(errs[(errs.y_true == 2) & (errs.y_pred != 2)])
print(f"\ndisease missed (called healthy) : {missed}")
print(f"false alarms  (healthy called sick): {false_alarm}")

CONFUSION MATRIX   (rows = true, cols = predicted)

                Early_blight   Late_blight       healthy
  Early_blight         994 ✓             5             1
   Late_blight             2         988 ✓            10
       healthy             0             3         149 ✓


PER-CLASS METRICS

              precision    recall  f1-score   support

Early_blight     0.9980    0.9940    0.9960      1000
 Late_blight     0.9920    0.9880    0.9900      1000
     healthy     0.9313    0.9803    0.9551       152

    accuracy                         0.9902      2152
   macro avg     0.9737    0.9874    0.9804      2152
weighted avg     0.9905    0.9902    0.9903      2152

pooled macro-F1 : 0.9804
per-fold mean   : 0.9804 ± 0.0063  (from summary.csv)

ERROR BREAKDOWN
total errors    : 21 / 2152
    Late_blight -> healthy        10
   Early_blight -> Late_blight     5
        healthy -> Late_blight     3
    Late_blight -> Early_blight    2
   Early_blight -> healthy         1

disease 

In [5]:
# 07 : Step 1: bootstrap + load fold 0 checkpoint
import sys, os, importlib
SCRIPTS = "/content/drive/MyDrive/0_potato_project_v1/scripts"

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path = [p for p in sys.path if p != SCRIPTS]
sys.path.insert(0, SCRIPTS)
importlib.invalidate_caches()

import numpy as np, pandas as pd, torch
import config as C, data as D, model as M

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_DIR = C.RESULTS / "runs" / "20260821_2337_baseline"
FOLD    = 0

D.setup_data()                                   # Grad-CAM opens real image files

net, meta = M.load_checkpoint(RUN_DIR / f"fold_{FOLD}" / "best.pt", device=DEVICE)
target_layer = net.features[-1]                  # last spatial layer, 7x7 grid

oof = pd.read_csv(RUN_DIR / f"fold_{FOLD}" / "oof_preds.csv")

print("checkpoint :", {k: meta[k] for k in ("fold", "epoch", "aug_mode", "img_size")})
print("held-out F1:", round(meta["metrics"]["macro_f1"], 4))
print("target layer:", type(target_layer).__name__,
      "->", target_layer[0].out_channels, "channels")
print("oof rows   :", len(oof), "| errors:", int((oof.y_pred != oof.y_true).sum()))
print("device     :", DEVICE, "| eval mode:", not net.training)

checkpoint : {'fold': 0, 'epoch': 1, 'aug_mode': 'baseline', 'img_size': 224}
held-out F1: 0.9701
target layer: Conv2dNormActivation -> 960 channels
oof rows   : 429 | errors: 7
device     : cpu | eval mode: True


In [6]:
# 07: Step 2: Grad-CAM
import torch.nn.functional as F


class GradCAM:
    """Class-discriminative heatmaps from the last spatial layer.

    Weights each feature map by how strongly the class score responds to it,
    sums them, keeps positive evidence only, and upsamples to image size.
    """

    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        self.h = [
            layer.register_forward_hook(
                lambda m, i, o: setattr(self, "acts", o.detach())),
            layer.register_full_backward_hook(
                lambda m, gi, go: setattr(self, "grads", go[0].detach())),
        ]

    def __call__(self, x, class_idx=None):
        """x: (1,3,H,W) normalised tensor. Returns (cam HxW in [0,1], probs, idx)."""
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)                          # fp32: AMP off deliberately
        probs = torch.softmax(logits.float(), 1)[0].detach().cpu().numpy()
        idx = int(logits.argmax(1)) if class_idx is None else int(class_idx)

        logits[0, idx].backward()

        w = self.grads.mean(dim=(2, 3), keepdim=True)   # importance per channel
        cam = F.relu((w * self.acts).sum(1, keepdim=True))   # positive evidence only
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear",
                            align_corners=False)[0, 0]

        cam = cam - cam.min()
        cam = cam / cam.max() if cam.max() > 0 else cam     # guard: all-zero map
        return cam.cpu().numpy(), probs, idx

    def close(self):
        for h in self.h:
            h.remove()


cam_engine = GradCAM(net, target_layer)
print("GradCAM attached to:", type(target_layer).__name__)
print("hooks registered   :", len(cam_engine.h))
print("note: model stays in eval mode; gradients flow but weights never update")

GradCAM attached to: Conv2dNormActivation
hooks registered   : 2
note: model stays in eval mode; gradients flow but weights never update


In [7]:
# 07 : Step 3: one image end-to-end
from PIL import Image

_, tf_eval = D.build_transforms(meta["aug_mode"])     # from checkpoint, not config
names = [C.IDX_TO_CLASS[i].replace("Potato___", "") for i in range(C.NUM_CLASSES)]


def load_tensor(rel_path):
    """Same preprocessing the model was evaluated under. Returns (1,3,224,224)."""
    with Image.open(C.DATA_ROOT / rel_path) as im:
        return tf_eval(im.convert("RGB")).unsqueeze(0).to(DEVICE)


row = oof[oof.y_pred == oof.y_true].iloc[0]
x = load_tensor(row.path)
cam, probs, idx = cam_engine(x)

print("image      :", row.path.split('/')[-1])
print("true       :", names[row.y_true], "| predicted:", names[idx])
print("probs      :", [round(float(p), 4) for p in probs])
print("csv probs  :", [round(row[f'p{i}'], 4) for i in range(3)])
print("match      :", np.allclose(probs, [row[f'p{i}'] for i in range(3)], atol=1e-3))
print()
print("cam shape  :", cam.shape, "| range:", f"[{cam.min():.3f}, {cam.max():.3f}]")
print("cam mean   :", f"{cam.mean():.3f}", "| non-zero:", f"{(cam > 0.01).mean():.1%}")

image      : 034959c1-f1e8-4a79-a6d5-3c1d14efa2f3___RS_Early.B 7136.JPG
true       : Early_blight | predicted: Early_blight
probs      : [1.0, 0.0, 0.0]
csv probs  : [np.float64(1.0), np.float64(0.0), np.float64(0.0)]
match      : True

cam shape  : (224, 224) | range: [0.000, 1.000]
cam mean   : 0.370 | non-zero: 91.6%


In [8]:
# 07 : Step 4: overlay + correct predictions, 3 per class
import matplotlib.pyplot as plt
import matplotlib.cm as cm_maps

MEAN = np.array(C.IMAGENET_MEAN).reshape(3, 1, 1)
STD  = np.array(C.IMAGENET_STD).reshape(3, 1, 1)


def to_image(x):
    """Undo ImageNet normalisation -> displayable HxWx3 array in [0,1]."""
    a = x[0].cpu().numpy() * STD + MEAN
    return np.clip(a, 0, 1).transpose(1, 2, 0)


def overlay(img, cam, alpha=0.45):
    heat = cm_maps.jet(cam)[..., :3]
    return np.clip((1 - alpha) * img + alpha * heat, 0, 1)


correct = oof[oof.y_pred == oof.y_true]
picks = [correct[correct.y_true == c].head(3) for c in range(C.NUM_CLASSES)]

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for r, sub in enumerate(picks):
    for i, (_, row) in enumerate(sub.iterrows()):
        x = load_tensor(row.path)
        cam, probs, idx = cam_engine(x)
        img = to_image(x)

        axes[r, i*2].imshow(img); axes[r, i*2].axis("off")
        axes[r, i*2].set_title(f"{names[row.y_true]}", fontsize=9)
        axes[r, i*2+1].imshow(overlay(img, cam)); axes[r, i*2+1].axis("off")
        axes[r, i*2+1].set_title(f"p={probs[idx]:.3f}  hot={((cam>0.5).mean()):.0%}",
                                 fontsize=9)

plt.suptitle(f"Grad-CAM — correct predictions, fold {FOLD} held-out", fontsize=13)
plt.tight_layout(); plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [9]:
# 07 :Step 5: misclassified images, predicted vs true class maps
errs = oof[oof.y_pred != oof.y_true].reset_index(drop=True)
n = len(errs)

fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
axes = axes.reshape(n, 3)

for r, (_, row) in enumerate(errs.iterrows()):
    x = load_tensor(row.path)
    cam_p, probs, _ = cam_engine(x, class_idx=row.y_pred)    # evidence for the answer given
    cam_t, _, _     = cam_engine(x, class_idx=row.y_true)    # evidence for the right answer
    img = to_image(x)

    axes[r, 0].imshow(img)
    axes[r, 0].set_title(f"true {names[row.y_true]}", fontsize=9)
    axes[r, 1].imshow(overlay(img, cam_p))
    axes[r, 1].set_title(f"said {names[row.y_pred]}  p={probs[row.y_pred]:.2f}", fontsize=9)
    axes[r, 2].imshow(overlay(img, cam_t))
    axes[r, 2].set_title(f"for {names[row.y_true]}  p={probs[row.y_true]:.2f}", fontsize=9)
    for c in range(3):
        axes[r, c].axis("off")

plt.suptitle(f"Grad-CAM — {n} errors, fold {FOLD} held-out", fontsize=13)
plt.tight_layout(); plt.show()

print("\nERROR DETAIL")
for _, row in errs.iterrows():
    print(f"  {names[row.y_true]:>13} -> {names[row.y_pred]:<13} "
          f"conf {row.confidence:.3f}   {row.path.split('/')[-1][:40]}")
print(f"\nmean confidence on errors  : {errs.confidence.mean():.3f}")
print(f"mean confidence on correct : {oof[oof.y_pred == oof.y_true].confidence.mean():.3f}")

Output hidden; open in https://colab.research.google.com to view.

In [12]:
# 06 : Step 5: confidence threshold sweep
import numpy as np

errs_mask = (oof.y_pred != oof.y_true).values
conf = oof.confidence.values
n_err = int(errs_mask.sum())
n_ok = int((~errs_mask).sum())

print(f"{'thresh':>7} {'flagged':>8} {'caught':>14} {'through':>8} {'correct flagged':>18}")
print("-" * 60)

for t in [0.50, 0.70, 0.80, 0.85, 0.90, 0.95, 0.99]:
    low = conf < t
    caught = int((low & errs_mask).sum())
    correct_flagged = int((low & ~errs_mask).sum())
    print(f"{t:>7.2f} {int(low.sum()):>8} {caught:>6}/{n_err:<7} "
          f"{n_err - caught:>8} {correct_flagged:>8} ({correct_flagged / n_ok:>6.2%})")

# the expensive error in an advisory context: diseased leaf called healthy
missed = ((oof.y_true != 2) & (oof.y_pred == 2)).values
print(f"\nMISSED DISEASES (n={int(missed.sum())}) — confidences:")
print("  ", sorted(np.round(conf[missed], 3).tolist()))
for t in [0.80, 0.90, 0.95]:
    print(f"  threshold {t:.2f} catches {int((conf[missed] < t).sum())}/{int(missed.sum())}")

 thresh  flagged         caught  through    correct flagged
------------------------------------------------------------
   0.50        1      0/21            21        1 ( 0.05%)
   0.70       10      6/21            15        4 ( 0.19%)
   0.80       23     15/21             6        8 ( 0.38%)
   0.85       24     16/21             5        8 ( 0.38%)
   0.90       36     18/21             3       18 ( 0.84%)
   0.95       43     20/21             1       23 ( 1.08%)
   0.99      107     20/21             1       87 ( 4.08%)

MISSED DISEASES (n=11) — confidences:
   [0.651, 0.673, 0.735, 0.744, 0.76, 0.771, 0.794, 0.811, 0.861, 0.915, 0.992]
  threshold 0.80 catches 7/11
  threshold 0.90 catches 9/11
  threshold 0.95 catches 10/11


In [13]:
%%writefile /content/drive/MyDrive/0_potato_project_v1/scripts/eval.py
"""Analysis of finished cross-validation runs: pooled metrics and Grad-CAM.

Reads artifacts written by train.py and never trains anything, so it runs in
seconds on CPU against any completed run directory.

    python eval.py results/runs/20260821_2337_baseline
"""
import argparse
from pathlib import Path

import matplotlib
matplotlib.use("Agg")                      # no display in a script context
import matplotlib.cm as cm_maps
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import config as C
import data as D
import model as M

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NAMES = [C.IDX_TO_CLASS[i].replace("Potato___", "") for i in range(C.NUM_CLASSES)]
MEAN = np.array(C.IMAGENET_MEAN).reshape(3, 1, 1)
STD = np.array(C.IMAGENET_STD).reshape(3, 1, 1)


# ── Pooling ───────────────────────────────────────────────────────────────
def pool_oof(run_dir, n_folds=C.N_FOLDS, verify=True):
    """Stack per-fold OOF predictions into one row-per-image frame.

    Every image appears exactly once, predicted by a model that never trained on
    it. A duplicate path means folds overlapped and every metric below is
    inflated, so it is checked rather than assumed.
    """
    run_dir = Path(run_dir)
    oof = pd.concat(
        [pd.read_csv(run_dir / f"fold_{k}" / "oof_preds.csv") for k in range(n_folds)],
        ignore_index=True)

    if verify:
        dups = len(oof) - oof.path.nunique()
        assert dups == 0, f"{dups} duplicate paths — folds overlap"
        probs = oof[[f"p{i}" for i in range(C.NUM_CLASSES)]].values
        assert np.allclose(probs.sum(1), 1), "probabilities do not sum to 1"

    return oof


# ── Metrics ───────────────────────────────────────────────────────────────
def report(oof, summary=None, verbose=True):
    """Confusion matrix, per-class metrics, error breakdown. Returns a dict."""
    y_true, y_pred = oof.y_true.values, oof.y_pred.values
    cm = confusion_matrix(y_true, y_pred, labels=range(C.NUM_CLASSES))
    errs = oof[oof.y_pred != oof.y_true]

    # in an advisory context these two are not symmetric: a missed disease
    # spreads, a false alarm costs one wasted inspection
    missed = int(((errs.y_true != 2) & (errs.y_pred == 2)).sum())
    false_alarm = int(((errs.y_true == 2) & (errs.y_pred != 2)).sum())

    out = {
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "accuracy": float((y_pred == y_true).mean()),
        "n_errors": len(errs),
        "disease_missed": missed,
        "false_alarms": false_alarm,
        "confusion": cm,
        "conf_correct": float(oof[oof.y_pred == oof.y_true].confidence.mean()),
        "conf_error": float(errs.confidence.mean()) if len(errs) else float("nan"),
    }

    if verbose:
        print("CONFUSION MATRIX   (rows = true, cols = predicted)\n")
        print(f"{'':>14}" + "".join(f"{n:>14}" for n in NAMES))
        for i, n in enumerate(NAMES):
            print(f"{n:>14}" + "".join(
                f"{cm[i, j]:>14}" if i != j else f"{str(cm[i, j]) + ' ✓':>14}"
                for j in range(C.NUM_CLASSES)))

        print("\n\nPER-CLASS METRICS\n")
        print(classification_report(y_true, y_pred, target_names=NAMES,
                                    digits=4, zero_division=0))
        print(f"pooled macro-F1 : {out['macro_f1']:.4f}")

        if summary is not None:
            s = pd.read_csv(summary) if not isinstance(summary, pd.DataFrame) else summary
            print(f"per-fold mean   : {s.test_macro_f1.mean():.4f} "
                  f"± {s.test_macro_f1.std():.4f}")

        print("\nERROR BREAKDOWN")
        print(f"total errors    : {len(errs)} / {len(oof)}")
        for (t, p), n in (errs.groupby(["y_true", "y_pred"]).size()
                          .sort_values(ascending=False).items()):
            print(f"  {NAMES[t]:>13} -> {NAMES[p]:<13} {n:>3}")
        print(f"\ndisease missed (called healthy)   : {missed}")
        print(f"false alarms  (healthy called sick): {false_alarm}")
        print(f"\nmean confidence on correct : {out['conf_correct']:.3f}")
        print(f"mean confidence on errors  : {out['conf_error']:.3f}")

    return out


# ── Grad-CAM ──────────────────────────────────────────────────────────────
class GradCAM:
    """Class-discriminative heatmaps from the last spatial layer.

    Weights each feature map by how strongly the class score responds to it,
    sums them, keeps positive evidence only, and upsamples to image size.

    Shows WHERE the model drew evidence, never WHAT feature it used -- a model
    keying on a whole-image colour cast produces on-leaf maps indistinguishable
    from one reading lesions. Spatial shortcuts only.
    """

    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        self.h = [
            layer.register_forward_hook(
                lambda m, i, o: setattr(self, "acts", o.detach())),
            layer.register_full_backward_hook(
                lambda m, gi, go: setattr(self, "grads", go[0].detach())),
        ]

    def __call__(self, x, class_idx=None):
        """x: (1,3,H,W) normalised. Returns (cam HxW in [0,1], probs, idx)."""
        self.model.zero_grad(set_to_none=True)
        logits = self.model(x)                       # fp32: AMP off deliberately
        probs = torch.softmax(logits.float(), 1)[0].detach().cpu().numpy()
        idx = int(logits.argmax(1)) if class_idx is None else int(class_idx)

        logits[0, idx].backward()

        w = self.grads.mean(dim=(2, 3), keepdim=True)          # channel importance
        cam = F.relu((w * self.acts).sum(1, keepdim=True))     # positive evidence
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear",
                            align_corners=False)[0, 0]

        cam = cam - cam.min()
        cam = cam / cam.max() if cam.max() > 0 else cam
        return cam.cpu().numpy(), probs, idx

    def close(self):
        for h in self.h:
            h.remove()


def load_fold(run_dir, fold=0):
    """Returns (net, meta, GradCAM, transform, oof) for one fold's checkpoint."""
    run_dir = Path(run_dir)
    net, meta = M.load_checkpoint(run_dir / f"fold_{fold}" / "best.pt", device=DEVICE)
    _, tf_eval = D.build_transforms(meta["aug_mode"])   # from checkpoint, not config
    oof = pd.read_csv(run_dir / f"fold_{fold}" / "oof_preds.csv")
    return net, meta, GradCAM(net, net.features[-1]), tf_eval, oof


def load_tensor(rel_path, tf_eval):
    with Image.open(C.DATA_ROOT / rel_path) as im:
        return tf_eval(im.convert("RGB")).unsqueeze(0).to(DEVICE)


def to_image(x):
    """Undo ImageNet normalisation -> displayable HxWx3 in [0,1]."""
    a = x[0].cpu().numpy() * STD + MEAN
    return np.clip(a, 0, 1).transpose(1, 2, 0)


def overlay(img, cam, alpha=0.45):
    heat = cm_maps.jet(cam)[..., :3]
    return np.clip((1 - alpha) * img + alpha * heat, 0, 1)


def plot_correct(engine, tf_eval, oof, fold=0, per_class=3, save=None):
    """Correct predictions, per_class rows deep. Background heat is the tell."""
    correct = oof[oof.y_pred == oof.y_true]
    picks = [correct[correct.y_true == c].head(per_class) for c in range(C.NUM_CLASSES)]

    fig, axes = plt.subplots(C.NUM_CLASSES, per_class * 2,
                             figsize=(3 * per_class * 2, 3 * C.NUM_CLASSES))
    axes = np.atleast_2d(axes)

    for r, sub in enumerate(picks):
        for i, (_, row) in enumerate(sub.iterrows()):
            x = load_tensor(row.path, tf_eval)
            cam, probs, idx = engine(x)
            img = to_image(x)

            axes[r, i * 2].imshow(img)
            axes[r, i * 2].set_title(NAMES[row.y_true], fontsize=9)
            axes[r, i * 2 + 1].imshow(overlay(img, cam))
            axes[r, i * 2 + 1].set_title(
                f"p={probs[idx]:.3f}  hot={(cam > 0.5).mean():.0%}", fontsize=9)
            for c in (i * 2, i * 2 + 1):
                axes[r, c].axis("off")

    fig.suptitle(f"Grad-CAM — correct predictions, fold {fold} held-out", fontsize=13)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=110, bbox_inches="tight")
    return fig


def plot_errors(engine, tf_eval, oof, fold=0, save=None):
    """Every error, with maps for both the predicted and the true class."""
    errs = oof[oof.y_pred != oof.y_true].reset_index(drop=True)
    if not len(errs):
        return None

    fig, axes = plt.subplots(len(errs), 3, figsize=(9, 3 * len(errs)))
    axes = axes.reshape(len(errs), 3)

    for r, (_, row) in enumerate(errs.iterrows()):
        x = load_tensor(row.path, tf_eval)
        cam_p, probs, _ = engine(x, class_idx=row.y_pred)     # evidence for the answer
        cam_t, _, _ = engine(x, class_idx=row.y_true)         # evidence for the truth
        img = to_image(x)

        axes[r, 0].imshow(img)
        axes[r, 0].set_title(f"true {NAMES[row.y_true]}", fontsize=9)
        axes[r, 1].imshow(overlay(img, cam_p))
        axes[r, 1].set_title(f"said {NAMES[row.y_pred]}  p={probs[row.y_pred]:.2f}",
                             fontsize=9)
        axes[r, 2].imshow(overlay(img, cam_t))
        axes[r, 2].set_title(f"for {NAMES[row.y_true]}  p={probs[row.y_true]:.2f}",
                             fontsize=9)
        for c in range(3):
            axes[r, c].axis("off")

    fig.suptitle(f"Grad-CAM — {len(errs)} errors, fold {fold} held-out", fontsize=13)
    fig.tight_layout()
    if save:
        fig.savefig(save, dpi=110, bbox_inches="tight")
    return fig


# ── Entry point ───────────────────────────────────────────────────────────
def analyse(run_dir, fold=0, figures=True, verbose=True):
    """Pool, report, and optionally write Grad-CAM figures into the run folder."""
    run_dir = Path(run_dir)
    oof = pool_oof(run_dir)
    summary = run_dir / "summary.csv"
    out = report(oof, summary=summary if summary.exists() else None, verbose=verbose)

    if figures:
        D.setup_data(verbose=verbose)          # Grad-CAM opens real image files
        net, meta, engine, tf_eval, fold_oof = load_fold(run_dir, fold)
        fig_dir = run_dir / "figures"
        fig_dir.mkdir(exist_ok=True)
        plot_correct(engine, tf_eval, fold_oof, fold,
                     save=fig_dir / f"gradcam_correct_fold{fold}.png")
        plot_errors(engine, tf_eval, fold_oof, fold,
                    save=fig_dir / f"gradcam_errors_fold{fold}.png")
        engine.close()
        plt.close("all")
        if verbose:
            print(f"\nfigures -> {fig_dir}")

    return oof, out


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("run_dir", type=str, help="path to a completed run directory")
    p.add_argument("--fold", type=int, default=0, help="fold to visualise")
    p.add_argument("--no-figures", action="store_true", help="metrics only")
    a = p.parse_args()
    analyse(a.run_dir, fold=a.fold, figures=not a.no_figures)


if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/0_potato_project_v1/scripts/eval.py


In [14]:
import importlib, sys
SCRIPTS = "/content/drive/MyDrive/0_potato_project_v1/scripts"
sys.path = [p for p in sys.path if p != SCRIPTS]
sys.path.insert(0, SCRIPTS)
importlib.invalidate_caches()

import config, data, model, eval as E
for m in (config, data, model, E):
    importlib.reload(m)

print(len(open(E.__file__).read().splitlines()), "lines\n")

RUN = config.RESULTS / "runs" / "20260821_2337_baseline"
oof = E.pool_oof(RUN)
print("pooled:", oof.shape, "\n")
out = E.report(oof, summary=RUN / "summary.csv")

275 lines

pooled: (2152, 8) 

CONFUSION MATRIX   (rows = true, cols = predicted)

                Early_blight   Late_blight       healthy
  Early_blight         994 ✓             5             1
   Late_blight             2         988 ✓            10
       healthy             0             3         149 ✓


PER-CLASS METRICS

              precision    recall  f1-score   support

Early_blight     0.9980    0.9940    0.9960      1000
 Late_blight     0.9920    0.9880    0.9900      1000
     healthy     0.9313    0.9803    0.9551       152

    accuracy                         0.9902      2152
   macro avg     0.9737    0.9874    0.9804      2152
weighted avg     0.9905    0.9902    0.9903      2152

pooled macro-F1 : 0.9804
per-fold mean   : 0.9804 ± 0.0063

ERROR BREAKDOWN
total errors    : 21 / 2152
    Late_blight -> healthy        10
   Early_blight -> Late_blight     5
        healthy -> Late_blight     3
    Late_blight -> Early_blight    2
   Early_blight -> healthy         

In [15]:
import importlib, sys
SCRIPTS = "/content/drive/MyDrive/0_potato_project_v1/scripts"
sys.path = [p for p in sys.path if p != SCRIPTS]
sys.path.insert(0, SCRIPTS)
importlib.invalidate_caches()
import config, eval as E
importlib.reload(config); importlib.reload(E)

print(len(open(E.__file__).read().splitlines()), "lines")
print("functions:", [f for f in ("pool_oof", "report", "threshold_sweep",
                                 "GradCAM", "plot_correct", "plot_errors", "analyse")
                     if hasattr(E, f)])

275 lines
functions: ['pool_oof', 'report', 'GradCAM', 'plot_correct', 'plot_errors', 'analyse']
